In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_2020_UGA.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2020_UGA.ID_001,0,ENSG00000000003,83,0.607222,Unknown
1,2020_UGA.ID_001,0,ENSG00000000005,0,0.000000,Unknown
2,2020_UGA.ID_001,0,ENSG00000000419,2489,68.417030,Unknown
3,2020_UGA.ID_001,0,ENSG00000000457,1221,5.885515,Unknown
4,2020_UGA.ID_001,0,ENSG00000000460,532,2.958027,Unknown


In [3]:
# Unlike 2019, raw_count has actual values in 2020; material is still all 'Unknown'
# Drop material before pivoting; keep raw_count aside — we pivot on tpm_count for consistency with 2019
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())

raw_count null fraction: 0.0
material unique values: <StringArray>
['Unknown']
Length: 1, dtype: str


In [4]:
# Drop material (entirely 'Unknown'); drop raw_count to match 2019 pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2020_UGA.ID_001,0,ENSG00000000003,0.607222
1,2020_UGA.ID_001,0,ENSG00000000005,0.000000
2,2020_UGA.ID_001,0,ENSG00000000419,68.417030
3,2020_UGA.ID_001,0,ENSG00000000457,5.885515
4,2020_UGA.ID_001,0,ENSG00000000460,2.958027


In [5]:
# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='ensembl_gene_id',
    values='tpm_count'
)
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()
print(df_pivot.shape)
df_pivot.head()

(96, 56416)


,participant_id,timepoint,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,...,ENSG00000280441,ENSG00000280443,ENSG00000280444,ENSG00000280445,ENSG00000280446,ENSG00000280449,ENSG00000280450,ENSG00000280451,ENSG00000280453,ENSG00000280454
0,2020_UGA.ID_001,0,0.607222,0.0,68.417030,5.885515,2.958027,445.721808,0.904403,36.064293,...,13.908368,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2020_UGA.ID_001,28,0.253107,0.0,61.740837,4.840439,2.259048,386.221818,1.228732,38.631147,...,13.932189,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2020_UGA.ID_005,0,0.340844,0.0,54.038282,3.512946,0.966797,297.316528,0.186410,17.828331,...,5.213905,0.0,0.18494,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2020_UGA.ID_005,28,0.218916,0.0,12.998795,2.091440,2.685836,124.016449,1.225569,16.458819,...,14.492502,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2020_UGA.ID_011,0,0.409859,0.0,37.957544,3.339057,0.564065,214.556388,1.335459,31.934511,...,10.529557,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_2020_UGA_cleaned.csv', index=False)